## Setup

Load the dataset and clean `TotalCharges` (blank for customers with zero tenure; cast to numeric and filled with 0).

In [1]:
import pandas as pd

telco = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')
telco['TotalCharges'] = telco['TotalCharges'].fillna(0)
telco['TotalCharges'] = telco['TotalCharges'].astype(float)

telco.head()

   customerID  gender  SeniorCitizen  ... MonthlyCharges TotalCharges  Churn
0  7590-VHVEG  Female              0  ...          29.85        29.85     No
1  5575-GNVDE    Male              0  ...          56.95      1889.50     No
2  3668-QPYBK    Male              0  ...          53.85       108.15    Yes
3  7795-CFOCW    Male              0  ...          42.30      1840.75     No
4  9237-HQITU  Female              0  ...          70.70       151.65    Yes

[5 rows x 21 columns]

## Business Question 1: Which contract types and tenure ranges have the highest churn rate?

Understanding whether month-to-month customers churn at a meaningfully higher rate than one- or two-year contract holders, and whether churn is concentrated in early tenure, helps prioritize retention efforts and evaluate whether incentivizing longer contracts would reduce churn.

In [2]:
contract_churn = telco.groupby('Contract')['Churn'].value_counts(normalize=True)
contract_churn

Contract        Churn
Month-to-month  No       0.572903
                Yes      0.427097
One year        No       0.887305
                Yes      0.112695
Two year        No       0.971681
                Yes      0.028319
Name: proportion, dtype: float64

**Finding:** churn rate scales sharply with contract length flexibility — **42.7%** of month-to-month customers churn, vs **11.3%** of one-year and only **2.8%** of two-year customers. Month-to-month customers are roughly **15x** more likely to churn than two-year customers. This is the single strongest churn signal in the dataset and suggests contract length is doing a lot of the retention work already — incentivizing upgrades from month-to-month to annual contracts (discounts, loyalty pricing) is likely the highest-leverage retention lever available.

## Business Question 2: How does monthly and total spend relate to churn?

Are higher-paying customers more or less likely to churn than lower-paying ones? This informs whether pricing/value perception is a churn driver and whether high-value customers need targeted retention offers.

In [3]:
# (a) average monthly spend by churn group
print(telco.groupby('Churn')['MonthlyCharges'].mean())
print()

# (b) average total spend (lifetime revenue) by churn group
print(telco.groupby('Churn')['TotalCharges'].mean())
print()

# (c) churn rate by price band -- do higher payers churn more?
telco['price_band'] = pd.cut(telco['MonthlyCharges'],
                              bins=[0, 35, 70, 120],
                              labels=['Low', 'Mid', 'High'])
print(telco.groupby('price_band')['Churn'].value_counts(normalize=True).unstack()['Yes'])

Churn
No     61.265124
Yes    74.441332
Name: MonthlyCharges, dtype: float64

Churn
No     2549.911442
Yes    1531.796094
Name: TotalCharges, dtype: float64

price_band
Low     0.108934
Mid     0.239420
High    0.353614
Name: Yes, dtype: float64


**Finding:** the two spend metrics point in opposite directions, and reconciling them matters. Churners pay **more per month** on average (**$74.44 vs $61.27** for non-churners), but have **lower total lifetime charges** (**$1,531.80 vs $2,549.91**) — because churners tend to leave earlier in their tenure, so they haven't accumulated as much total spend. Breaking `MonthlyCharges` into price bands confirms the monthly-spend relationship directly: churn rate rises from **10.9%** in the Low band, to **23.9%** Mid, to **35.4%** High. Higher monthly bills are associated with meaningfully higher churn risk — `TotalCharges` is mostly a proxy for tenure here, not an independent driver, so `MonthlyCharges`/price band is the more useful signal for a churn model or a pricing/retention conversation.

## Business Question 3: Which services or add-ons are associated with lower churn?

Do customers with add-ons like online security, tech support, or device protection churn less than those without? This suggests whether bundling or promoting these services could be an effective retention lever.

In [4]:
addons = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in addons:
    rates = telco.groupby(col)['Churn'].value_counts(normalize=True).unstack()['Yes']
    print(f"Churn rate by {col}: with = {rates['Yes']:.0%}   without = {rates['No']:.0%}")

Churn rate by OnlineSecurity: with = 15%   without = 42%
Churn rate by OnlineBackup: with = 22%   without = 40%
Churn rate by DeviceProtection: with = 23%   without = 39%
Churn rate by TechSupport: with = 15%   without = 42%
Churn rate by StreamingTV: with = 30%   without = 34%
Churn rate by StreamingMovies: with = 30%   without = 34%


**Finding:** every add-on is associated with lower churn, but the effect size varies a lot by add-on. `OnlineSecurity` and `TechSupport` show the biggest gap — churn drops from **42%** without to **15%** with, roughly a **3x** reduction. `OnlineBackup` (40% -> 22%) and `DeviceProtection` (39% -> 23%) show a similar but slightly smaller pattern. `StreamingTV` and `StreamingMovies` show the weakest relationship (34% -> 30%), suggesting entertainment add-ons track more with usage/engagement than with retention itself. This ranks `OnlineSecurity` and `TechSupport` as the strongest bundling/promotion candidates for a retention campaign — though since add-on adoption correlates with contract type and tenure, a proper causal read would need a model that controls for those confounders rather than this bivariate view alone.